# Spilled Energy — TriviaQA on a T4

Run the cells **in order**. Each gate is designed to fail fast and loudly:
a bad prompt format, a silent model swap, or drifting generations all stop the
notebook rather than producing numbers that look fine and mean nothing.

Two runs, deliberately: the **primary baseline table** and the **ablation
ladder**. Both save to Drive as they finish, and cell 11 asserts they saw
byte-identical generations — otherwise the two tables are not comparable.

**Runtime → Change runtime type → T4 GPU** before starting.

## 1. Confirm you actually got a T4

Colab hands out different accelerators. Check before spending an hour.

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free,driver_version',
                      '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU'
name = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info()
print(f'device    : {name}')
print(f'VRAM free : {free/1e9:.2f} GB / {total/1e9:.2f} GB')
if 'T4' not in name:
    print(f'\nWARNING: expected a T4, got {name!r}. Results stay valid; timings differ.')

## 2. Mount Drive

Every run writes **straight to Drive**. lm-polygraph saves the manager inside a
`finally:` block, so a run that raises still leaves its results behind, and each
run lands before the next begins — a disconnect costs one run, not all of them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_OUT = Path('/content/drive/MyDrive/spilled_energy/runs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
DRIVE = DRIVE_OUT.as_posix()   # posix form, for shell interpolation
print('results ->', DRIVE)

## 3. Clone the experiments branch

`spilled-energy-experiments` = the PR branch **plus** `harness/` and this notebook.
The PR is opened from `spilled-energy`, which holds only the StatCalculator, the
Estimator, the tests and the configs.

In [ ]:
REPO_URL = 'https://github.com/neuezeldaa/lm-polygraph'
BRANCH   = 'spilled-energy-experiments'
REPO     = '/content/lm-polygraph'

import os
if not os.path.isdir(REPO):
    !git clone --branch $BRANCH $REPO_URL $REPO
%cd $REPO
!git log -1 --oneline

## 4. Install, and set PYTHONPATH explicitly

`PYTHONPATH` is set here rather than left to chance: the ablation ladder loads
`harness.pooled_baseline` by dotted path from inside a `polygraph_eval`
subprocess, which does not inherit the notebook's `sys.path`.

In [ ]:
!pip install -q -e .
!pip install -q -r harness/requirements-repro.txt

import os, sys, sysconfig
os.environ['PYTHONPATH'] = REPO + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['PATH'] = os.environ['PATH'] + os.pathsep + sysconfig.get_path('scripts')
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('PYTHONPATH =', os.environ['PYTHONPATH'])
!which polygraph_eval || echo 'NOTE: not on PATH; run_baselines.py prints a fallback'

## 5. Fail-fast import check

Five seconds here beats discovering a broken import inside a subprocess after the
3B model has loaded. Also proves the subprocess itself can resolve the dotted path.

In [ ]:
import subprocess, sys, os

from harness.pooled_baseline import PooledBaseline          # in-process
from lm_polygraph.estimators import SpilledEnergy
from lm_polygraph.stat_calculators import EnergyCalculator
print('in-process imports OK:', str(PooledBaseline(score='log_likelihood', pooling='max')))

# the path that actually matters: a fresh subprocess, as polygraph_eval will be
r = subprocess.run([sys.executable, '-c',
                    'from lm_polygraph.utils.factory_estimator import FactoryEstimator;'
                    'e=FactoryEstimator()("harness.pooled_baseline",'
                    '{"score":"log_likelihood","pooling":"max"});print("subprocess OK:",e)'],
                   capture_output=True, text=True, env=dict(os.environ))
print(r.stdout.strip() or r.stderr.strip())
assert r.returncode == 0, 'subprocess cannot import harness.* -- fix PYTHONPATH before running anything'

## 6. Model provenance

Printed before anything loads the model. `--expect` makes a silent model swap a
hard failure rather than a footnote. Confirm `model.path`, `dtype` and `device`
against the report.

In [ ]:
!python harness/provenance.py --config configs/stage1/eval_triviaqa_qwen.yaml --expect Qwen/Qwen2.5-3B-Instruct

## 7. Unit tests (CPU, seconds)

Cheap proof the install is sane before any long run.

In [ ]:
!python -m pytest test/test_spilled_energy.py -q

## 8. Dev run, n=150 — the gates fire here

This one short run does four jobs:

1. **Accuracy gate** — hard-fails outside 10–90% exact match. Outside that band
   PRR is noise and every downstream number is meaningless. The usual cause is
   prompt format.
2. **Sign check** — a wrong sign shows up as a large *negative* normalized PRR
   (~-0.7), which reads as a broken method rather than an inverted score.
3. **Answer-span validation** (cell 9).
4. **Runtime measurement** (cell 10).

If the accuracy gate fails, **stop and fix the prompt** — do not widen the band.

In [ ]:
cmd = ('python harness/run_baselines.py'
       ' --config configs/stage1/eval_triviaqa_qwen.yaml'
       f" --save-dir '{DRIVE}/dev_n150'"
       ' --samples 150 --n-boot 0'
       ' --expect-model Qwen/Qwen2.5-3B-Instruct')
print(cmd)
!{cmd}

## 9. Validate the answer-span assumption

The ablation ladder defines the answer window as the whole generation. That is
only defensible if the generation really is a short answer — measured, not
asserted. Reports single-line fraction, length distribution, ceiling-truncation
rate and exact-match rate.

**Re-run this for CoQA.** The assumption may not transfer.

In [ ]:
cmd = ('python harness/validate_answer_span.py'
       f" --save-dir '{DRIVE}/dev_n150'"
       ' --config configs/stage1/eval_triviaqa_qwen.yaml'
       ' --max-new-tokens 20')
!{cmd}

## 10. How long will the real runs take?

Projected from the measured n=150 pass, **before** committing to the long runs.
Linear in n, and it double-counts fixed startup, so it slightly overestimates.

In [ ]:
cmd = ('python harness/estimate_runtime.py'
       f" --from '{DRIVE}/dev_n150'"
       ' --label baselines_n1000 --to-n 1000')
!{cmd}
print('\nNOTE: the ladder run has fewer estimators but the same generation cost,')
print('so budget roughly the same again for run B.')

## 11. Run A — primary baseline table, n=1000

The mechanically derived `single_pass_cheap` + `single_pass_plus_aux_model` tiers
**and** all Spilled Energy variants, in one `UEManager` — so generations and
generation settings are identical by construction.

If the sign check flagged a variant as inverted, set `cfg: {sign: -1}` in
`configs/stage1/estimators/stage1_baselines.yaml` and commit. Do not patch it here.

In [ ]:
cmd = ('python harness/run_baselines.py'
       ' --config configs/stage1/eval_triviaqa_qwen.yaml'
       f" --save-dir '{DRIVE}/A_baselines_n1000'"
       ' --n-boot 1000'
       ' --expect-model Qwen/Qwen2.5-3B-Instruct')
print(cmd)
!{cmd}

## 12. Run B — ablation ladder, n=1000

Same window, same three poolings on every rung, so adjacent rungs differ by
exactly one ingredient: pooled log-likelihood → E^l → E^m → ΔE → ΔE_s.

Run A is already saved to Drive before this starts.

In [ ]:
cmd = ('python harness/run_baselines.py'
       ' --config configs/stage1/eval_triviaqa_ladder.yaml'
       f" --save-dir '{DRIVE}/B_ladder_n1000'"
       ' --n-boot 1000'
       ' --expect-model Qwen/Qwen2.5-3B-Instruct')
print(cmd)
!{cmd}

## 13. Are the two tables comparable? — hard gate

Greedy decoding at a fixed seed *should* make the two runs byte-identical, but
they resolve different stat calculators, and fp16 reductions are not associative.
So it is **verified, not trusted**: sha256 of the generations and the full
quality vector must match exactly.

If this fails, the two tables are about different generations and must not be
placed side by side — the fix is to make the configs agree on
`output_attentions` and re-run, not to proceed.

In [ ]:
cmd = ('python harness/check_run_consistency.py'
       f" --a '{DRIVE}/A_baselines_n1000'"
       f" --b '{DRIVE}/B_ladder_n1000'"
       ' --label-a baselines --label-b ladder')
!{cmd}

## 14. The reported tables

Primary metric is **normalized PRR@0.5** with bootstrap CIs. Everything on Drive,
so tables can be rebuilt offline on CPU with
`python harness/run_baselines.py --skip-run --save-dir <dir>`.

In [ ]:
from IPython.display import Markdown, display
for tag, label in [('A_baselines_n1000', 'PRIMARY BASELINE TABLE'),
                   ('B_ladder_n1000',   'ABLATION LADDER')]:
    p = DRIVE_OUT / tag / 'prr_0.5_table.md'
    display(Markdown(f'# {label}'))
    display(Markdown(p.read_text() if p.exists() else f'_missing: {p}_'))

In [ ]:
import torch, transformers, datasets, sys
print('python      ', sys.version.split()[0])
print('torch       ', torch.__version__)
print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)
print('device      ', torch.cuda.get_device_name(0))
!git rev-parse HEAD